From Fakeddit paper: "In addition, we used the BERT model. BERT achieves state-of-the-art results on many classification tasks, including Q&A and named entity recognition. To obtain fixed-length BERT embedding vectors, we used the bert-as-service(Xiao, 2018) tool, to map variable-length text/sentences into a 768 element array for each Reddit submission title. For our experiments, we utilized the pretrained BERT-Large, Uncased model."

Paper achieves around 82% accurcy on 3-way labels, so I should aim for the same

Use original BERT paper (Devlin et al., 2019) for fine-tuning procedure. Can be found in appendix A3:

Batch size: 16 or 32
Num epochs: 2, 3, or 4
Select best learning rate from 2, 3, 4 or 5 e-5 on val set
Dropout: 0.1
Large datasets (>100K labelled examples) far less sensitive to hyperparameter choice than small.
Fine-tuning is fast - best to run an exhaustive search across the above hyperparameters on the train/val set and see which is best.
Bert github code provides more set up details (google-research/bert):

AdamW optimiser (β1=0.9, β2=0.999, ε=1e-6)
Weight decay = 0.01
Warmup ratio (10% of total steps in published code)
(Mouratidis et al., 2025) - good paper for supporting claim that fine-tined BERT is current best practice for fake new detection. Not so good for fine-tuning methodology but they do have some interesting stuff to take into account:

They use Matthews Correlation Coefficient (MCC) and ROC_AUC alongside classic accuracy and macro F1 - good for datasets with a class imbalance (like mine).
Find that non-stemmed text performs better than stemmed and that unigrams are sufficient
This method is a bit outdated. Instead, going to fine tune bert-base-uncased end-to-end via Hugging Face Transformers library. Current best practice for BERT-based text classification.

Why bert-base-uncased?

Save GPU - BERT Large requires more GPU< likely for minimal gain. Use BERT-base instead
Uncased - using Reddit titles with inconsistent casing. Since Fakeddit's clean_title col is already lowercased, uncased makes the most sense.
Example explanation: "We use bert-base-uncased (Devlin et al., 2019: L=12, H=768, A=12, 110M parameters) rather than BERT-Large, balancing classification performance against training time within the project's compute budget. Devlin et al. (2019, §5.2) report that BERT-Large outperforms Base on most tasks but at substantially higher computational cost; for our classification setting on short titles, base-size models are standard in recent comparable work (Mouratidis et al., 2025; numerous HuggingFace baselines)."

Need to add justification from literature for using GradScaler

### 1. Setup

In [1]:
!pip install --user -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 133.9 MB/s  0:00:00
  Using cached accelerate-1.13.0-py3-none-any.whl.metadata (19 kB)
  Using cached aiohappyeyeballs-2.6.2-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiohttp-3.14.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (8.3 kB)
  Using cached anyio-4.13.0-py3-none-any.whl.metadata (4.5 kB)
  Using cached appnope-0.1.4-py2.py3-none-any.whl.metadata (908 bytes)
  Using cached async_lru-2.3.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached attrs-26.1.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached babel-2.18.0-py3-none-any.whl.metadata (2.2 kB)
  Using cached bertopic-0.17.4-py3-none-any.whl.metadata (24 kB)
  Using cached certifi-2026.5.20-py3-none-any.whl.metadata (2.5 kB)
  Using cached charset_normalizer-3.4.7-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (40 kB)
  Using cached click-8.4.1-py3-none-any.whl.metadata (2

In [ ]:
import os
import numpy as np
import pandas as pd
import json
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from datasets import Dataset
import itertools
from sklearn.metrics import matthews_corrcoef, accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW

import time
from pathlib import Path

import warnings

import torch.nn.functional as F

/opt/conda/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[HAMI-core Msg(137:139929406410048:libvgpu.c:839)]: Initializing.....


In [2]:
warnings.filterwarnings('ignore')

In [19]:
data_dir = os.path.join('../', 'data', 'processed', 'US')

train_path = os.path.join(data_dir, 'multimodal_train.tsv')
val_path = os.path.join(data_dir, 'multimodal_validate.tsv')
test_path = os.path.join(data_dir, 'multimodal_test_public.tsv')

train_df = pd.read_csv(train_path, sep='\t')
val_df = pd.read_csv(val_path, sep='\t')
test_df = pd.read_csv(test_path, sep='\t')


In [6]:
MODEL_NAME = 'bert-base-uncased'
NUM_LABELS = 3
MAX_LEN = 32 # from EDA, at least 75% of titles are under 10 words
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

tokeniser = AutoTokenizer.from_pretrained(MODEL_NAME)

[HAMI-core Msg(137:139929406410048:libvgpu.c:855)]: Initialized


Using device: cuda


In [7]:
def tokenise(batch):
    return tokeniser(batch['clean_title'], truncation=True, padding='max_length', max_length=MAX_LEN)

def prepare_dataset(df):
    """Converts df into dataset ready for torch"""
    ds = Dataset.from_pandas(df, preserve_index=False)
    ds = ds.map(tokenise, batched=True)
    ds = ds.rename_column('3_way_label', 'labels')
    keep_cols = {'input_ids', 'attention_mask', 'labels'}
    ds = ds.remove_columns([c for c in ds.column_names if c not in keep_cols])
    ds.set_format('torch')
    return ds

In [8]:
# small subset dataset to enable faster grid training
SUBSAMPLE_SIZE = 100_000
train_df_small, _ = train_test_split(
    train_df,
    train_size=SUBSAMPLE_SIZE,
    stratify=train_df["3_way_label"],
    random_state=42,
)

In [20]:
train_small_ds = prepare_dataset(train_df_small)
train_full_ds = prepare_dataset(train_df)
val_ds = prepare_dataset(val_df)
test_ds = prepare_dataset(test_df)

print(f"Train (subsample): {len(train_small_ds):,}")
print(f"Train (full): {len(train_full_ds):,}")
print(f"Val:   {len(val_ds):,}")


Map: 100%|██████████| 59319/59319 [00:02<00:00, 28661.60 examples/s]

Train (subsample): 100,000
Train (full): 564,000
Val:   59,342


### 2. Training and evaluation functions

Using torch (like in Neural Networks lectures)
- Each item has 3 tensors: input_ids, attention_mask, and lables from toeknise and prepare_dataset functions 
- Batch is formed of dicts, so unpack dict and pass items to model. 

In [11]:
def train_one_epoch(model, loader, optimiser, scheduler, scaler=None):
    """Forward pass over one epoch. Returns avg training loss"""
    model.train()
    total_loss = 0.0

    progress_bar = tqdm(loader, desc="Training Batches", leave=False) # leave=False clears bar once epoch has finished

    for batch in progress_bar: 
        batch = {k: v.to(device) for k, v in batch.items()}    # move batch to device
        optimiser.zero_grad()   # clear gradients from prev step

        if scaler is not None:
            with torch.amp.autocast('cuda'):
                loss = model(**batch).loss
            scaler.scale(loss).backward()          # Compute scaled gradients.
            scaler.step(optimiser)
            scaler.update()
        else:
            loss = model(**batch).loss  # ** unpacks the dict as keyword arguments
            loss.backward() # Compute gradients of loss
            optimiser.step()

        scheduler.step()    # Advance lr schedule
    
        total_loss += loss.item()

    return total_loss / len(loader) # return avg loss for epoch

include MCC as an evaluation metric because it is good for when classes are imbalanced. MCC takes into account all four quadrants of confusion matrix, across all classes. Only gives a high score if the model performs well across the whole matrix. 

In [12]:
# evaluation metrics
def evaluate(model, loader):
    """"Evaluate model. Returns accuraxy, macro F1, MCC, per-class precision and recall, confusion matrix"""
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in loader:
            labels = batch.pop('labels')    # Remove labels before assessing batch
            batch = {k: v.to(device) for k, v in batch.items()} 
            preds = model(**batch).logits.argmax(dim=-1).cpu().numpy()  # move back to cpu so can convert to numpy
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    
    report = classification_report(all_labels, all_preds, output_dict=True)

    return {
        'accuracy': accuracy_score(all_labels, all_preds),
        'macro_f1': f1_score(all_labels, all_preds, average='macro'),
        'MCC': matthews_corrcoef(all_labels, all_preds),
        'per_class': {k: v for k, v in report.items()},
        'confusion_matrix': confusion_matrix(all_labels, all_preds).tolist()    # need to convert to list so can be saved in json 
    }

Using 0.1 warmup ratio and 0.01 weight decay as recommended in BERT literature

In [ ]:
# full training loop

def train_model(train_ds, val_ds, batch_size, learning_rate, num_epochs, weight_decay=0.01, warmup_ratio=0.1, use_fp16=True, save_path=None):
    """Trains BERT for 'num_epochs'. Stores model with best F1"""

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    eval_loader = DataLoader(val_ds, batch_size=64, shuffle=False)

    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS).to(device)

    optimiser = AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    total_steps = len(train_loader) * num_epochs
    warmup_steps = int(warmup_ratio * total_steps)
    scheduler = get_linear_schedule_with_warmup(
        optimiser,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    # GradScaler enables mixed-precision training, which halves memory and speeds up GPU runs
    scaler = torch.amp.GradScaler('cuda') if (use_fp16 and device.type == 'cuda') else None

    best_f1 = -1.0
    best_epoch = -1
    best_state = None   # CPU copy of best model weights
    epoch_history = []

    for epoch in range(1, num_epochs + 1):

        print(f'Epoch {epoch} / {num_epochs}')

        start_time = time.perf_counter()
        
        train_loss = train_one_epoch(model, train_loader, optimiser, scheduler, scaler)
        val_metrics = evaluate(model, eval_loader)

        end_time = time.perf_counter()

        print(f'Train loss = {train_loss:.3f}\n'
              f'val F1 = {val_metrics["macro_f1"]:.3f}\n'
              f'val acc = {val_metrics["accuracy"]:.3f}\n'
              f'Duration = {end_time - start_time:.3f}s\n'
              )
        
        epoch_history.append({
            'epoch': epoch,
            'train_loss': train_loss,
            'duration_s': end_time - start_time,
            **val_metrics
        })

        if val_metrics['macro_f1'] > best_f1:           # Save best epoch so far
            best_f1 = val_metrics['macro_f1']
            best_epoch = epoch
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        
    model.load_state_dict(best_state)               # Restore best model state before continuing 

    if save_path:
        save_path = Path('.') / save_path
        Path(save_path).mkdir(parents=True, exist_ok=True)
        model.save_pretrained(save_path)
        tokeniser.save_pretrained(save_path)
    

    return {
        "best_epoch":    best_epoch,
        "best_macro_f1": best_f1,
        "epoch_history": epoch_history,
    }
        

### 3.1 Grid search on 100K sample
As recommended in BERT appendix A3, can do an exhaustive grid search over their batch size and lr hyperparamter recommendations

use train_df_small for the grid search and train_df (the full set) for the final training run

Do on the smaller training subset (to save on GPU). Then can run the full training loop on the pre-tuned configuration.

In [11]:
def save_results(results, path='results/bert_grid_search.json'):
    path = Path('.') / path
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w') as f:
        json.dump(results, f, indent=2, default=str)
    print(f'Saved {len(results)} results to {path}')
    

In [12]:
BATCH_SIZES = [32, 16] # run fastest first
LEARNING_RATES = [2e-5, 3e-5, 5e-5] # most stable first
MAX_EPOCHS = 4

grid_results = []

for batch_size, lr in itertools.product(BATCH_SIZES, LEARNING_RATES):
    loop_name = f'bs_{batch_size}_lr_{lr}'
    print(f'training run: {loop_name}')

    save_path = Path('.') / 'models' / 'bert_grid' / loop_name / 'best'

    result = train_model(
        train_small_ds,
        val_ds,
        batch_size, 
        lr,
        num_epochs=MAX_EPOCHS,
        save_path=save_path,  # save checkpoint per run
    )

    grid_results.append({
        'run_name': loop_name,
        'batch_size': batch_size, 
        'learning_rate': lr,
        **result
    })

    # save results after each loop
    save_results(grid_results)

    # print running leaderboard to monitor progress
    interim_df = pd.DataFrame(grid_results).sort_values('best_macro_f1', ascending=False)
    print(f"\nLeaderboard so far ({len(grid_results)}/6 runs):")
    print(interim_df[['run_name', 'best_epoch', 'best_macro_f1']].to_string(index=False))


# sort runs by validation f1 and print summary table
grid_df = pd.DataFrame(grid_results).sort_values('best_macro_f1', ascending=False)
print('GRID SEARCH RESULTS')
print(grid_df[["run_name", "best_epoch", "best_macro_f1"]].to_string(index=False))

best = grid_df.iloc[0]
print(f"\nBest hyperparameters: batch={best['batch_size']}, "
      f"lr={best['learning_rate']:.0e}, "
      f"epochs={best['best_epoch']}")      

training run: bs_32_lr_2e-05


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8191.12it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

Epoch 1 / 4


Training Batches:   0%|          | 0/3125 [00:00<?, ?it/s]/tmp/ipykernel_911/1786139231.py:13: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
                                                                     

Train loss = 0.432
val F1 = 0.854
val acc = 0.865
Duration = 213.680s

Epoch 2 / 4


...

Train loss = 0.133
val F1 = 0.855
val acc = 0.865
Duration = 340.876s

Epoch 4 / 4


Training Batches:   0%|          | 0/6250 [00:00<?, ?it/s]/tmp/ipykernel_911/1786139231.py:13: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
                                                                     

Train loss = 0.051
val F1 = 0.855
val acc = 0.863
Duration = 342.712s



Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.11it/s]

Saved 6 results to results/bert_grid_search.json

Leaderboard so far (6/6 runs):
      run_name  best_epoch  best_macro_f1
bs_32_lr_3e-05           2       0.866156
bs_16_lr_2e-05           2       0.863790
bs_32_lr_2e-05           4       0.860996
bs_16_lr_3e-05           2       0.860348
bs_16_lr_5e-05           4       0.855283
bs_32_lr_5e-05           3       0.854265
GRID SEARCH RESULTS
      run_name  best_epoch  best_macro_f1
bs_32_lr_3e-05           2       0.866156
bs_16_lr_2e-05           2       0.863790
bs_32_lr_2e-05           4       0.860996
bs_16_lr_3e-05           2       0.860348
bs_16_lr_5e-05           4       0.855283
bs_32_lr_5e-05           3       0.854265

Best hyperparameters: batch=32, lr=3e-05, epochs=2


### 3.2 Training on full training set

Re-train from scratch using the optimal hyperparameters found by grid search

In [16]:
best = pd.read_json('../results/bert_grid_search.json').sort_values('best_macro_f1', ascending=False).iloc[0]

final_result = train_model(
    train_full_ds,
    val_ds,
    batch_size=int(best['batch_size']),
    learning_rate=float(best["learning_rate"]),
    num_epochs=int(best["best_epoch"]),
    save_path="../models/bert_final/best"
)

print(f"\nFinal val macro-F1: {final_result['best_macro_f1']:.3f}")
print(f"Best epoch: {final_result['best_epoch']}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 225.36it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoin

Epoch 1 / 2


Train loss = 0.338
val F1 = 0.882
val acc = 0.887
Duration = 958.714s

Epoch 2 / 2


Train loss = 0.216
val F1 = 0.891
val acc = 0.895
Duration = 1011.745s



Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.06it/s]


Final val macro-F1: 0.891
Best epoch: 2


### 4 Test set evaluation

In [22]:
test_ds = prepare_dataset(test_df)
test_loader = DataLoader(test_ds, batch_size=64)
model = AutoModelForSequenceClassification.from_pretrained('../models/bert_final/best').to(device)
test_metrics = evaluate(model, test_loader)
print(f'Test metrics: {test_metrics}')

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 8830.58it/s]


Test metrics: {'accuracy': 0.8948734806722972, 'macro_f1': 0.8898748596662452, 'MCC': 0.7926464117335675, 'per_class': {'0': {'precision': 0.8716382455247307, 'recall': 0.8741226017781937, 'f1-score': 0.8728786559333914, 'support': 23507.0}, '1': {'precision': 0.9138790035587189, 'recall': 0.8605898123324397, 'f1-score': 0.886434242319641, 'support': 1492.0}, '2': {'precision': 0.9100465928945836, 'recall': 0.9105769230769231, 'f1-score': 0.9103116807457035, 'support': 34320.0}, 'accuracy': 0.8948734806722972, 'macro avg': {'precision': 0.8985212806593443, 'recall': 0.8817631123958521, 'f1-score': 0.8898748596662452, 'support': 59319.0}, 'weighted avg': {'precision': 0.8949224831673082, 'recall': 0.8948734806722972, 'f1-score': 0.8948771108373318, 'support': 59319.0}}, 'confusion_matrix': [[20548, 37, 2922], [41, 1284, 167], [2985, 84, 31251]]}


### 5. Veracity probability extraction for Pipeline B

Need to extract softmax probabilities for each class in order to use them as features in the virality predictor pipeline


In [ ]:
def extract_veracity_probabilities(ds, batch_ize=64):
    """
    Run inference on a prepared torch Dataset and return softmax probabilities.
    ds should be output of prepare_dataset() — same pipeline as training.
    """

    loader = DataLoader(ds, batch_size=batch_size, shuffle=False)
    all_probs = []

    model.eval()

    with torch.no_grad():
        for batch in tqdm(loader, desc='extracting probabilities'):
            labels = batch.pop('labels')
            batch = {k: v.to(device) for k, v, in batch.itmes()}
            logits = model(**batch).logits
            probs = F.softmax(logits, dim=1).cpu().numpy()
            all_probs.extend(probs)

    return pd.DataFrame(
        all_probs,
        columns=['p_true', 'p_fake_true_text', 'p_fake_false_text']
    )


In [ ]:
for name, df, ds in [('train', train_df, train_full_ds),
    ('val',   val_df,   val_ds),
    ('test',  test_df,  test_ds)]:

    print(f'Processing {name} ({len(df)} rows)')
    start = time.perf_counter()

    prob_df = extract_veracity_probabilities(ds)
    prob_df.index = df.index # need to align index before joining dfs

    for col in prob_df.columns:
        df[col] = prob_df[col]

    print(f'Finished in {time.perf_counter() - start:.1f}s')


In [ ]:
# checks on train df to ensure probabilities extracted correctly

# 1. probabilities sum to 1 in every row 
prob_cols = ['p_true', 'p_fake_true_text', 'p_fake_false_text']
row_sums = train_df[prob_cols].sum(axis=1).round(3)
print(f'Row sums all equal 1: {(row_sums == 1.0).all()}')

# 2. Highest prob class must match argmax prediction form evaluate()
train_df['predicted_label'] = train_df[prob_cols].values.argmax(axis=1)

# 3. check specific subreddits for expected results: propagandaposters should have high p_fake_true_text
prop_mask = train_df['subreddit'] == 'propagandaposters'
print(f'\nMean probabilities for propagandaposters (expect high p_fake_true_text):')
print(train_df.loc[prop_mask, prob_cols].mean().round(3))

# 4. check neutralnews should have high p_true
news_mask = train_df['subreddit'] == 'neutralnews'
print(f'\nMean probabilities for neutralnews (expect high p_true):')
print(train_df.loc[news_mask, prob_cols].mean().round(3))


In [ ]:
# save enriched dfs for pipeline B

out_dir = Path('../data/processed/US/with_veracity_probs')
out_dir.mkdir(parents=True, exist_ok=True)

train_df.to_csv(out_dir / 'train_with_probs.csv', index=False)
val_df.to_csv(out_dir / 'val_with_probs.csv', index=False)
test_df.to_csv(out_dir / 'test_with_probs.csv', index=False)

print(f'\nSaved enriched datasets to {out_dir}')